<center>

$\Huge \textbf{Universidad Nacional Autónoma de México}$  
$\Huge \textbf{Facultad de Ciencias}$  
<p align='center'>
  <img src='https://www.icat.unam.mx/wp-content/uploads/2021/11/Copia-de-LogoUNAM.-Azul.-Fondo-transparente.png' alt='UNAM' width='200'/>
</p>

<hr style='height:3px; background-color:#0B6E4F; border:none;'/>

$\LARGE \textbf{Inteligencia Artificial}$  

$\Large \textit{Algoritmo de Teoría — XGBoost (eXtreme Gradient Boosting)}$  

\begin{array}{rl}
\textbf{Docente:} & \text{Dra. Jessica Sarahi Méndez Rincón} \\[6pt]
\textbf{Ayudante de laboratorio:} & \text{Diego Eduardo Peña Villegas} \\[6pt]
\textbf{Alumna:} & \text{Marisol Luna Méndez} \\[6pt]
\textbf{Fecha:} & \text{2026}
\end{array}

</center>

---
# 1. Descripción del Algoritmo

## 1.1 Relación con Gradient Boosting

XGBoost **no es un algoritmo nuevo desde cero**. Es una implementación **optimizada y regularizada** del Gradient Boosting tradicional. Si Gradient Boosting y XGBoost son «padre e hijo», XGBoost es la versión con esteroides: mucho más rápida y con mecanismos que evitan el sobreajuste.

## 1.2 Descripción Narrativa

Si el Gradient Boosting es un aprendiz que corrige errores, XGBoost es ese mismo aprendiz con un **sistema de alto rendimiento y un estricto sentido de la autodisciplina**. Sus tres pilares diferenciadores son:

**Regularización extrema:** A diferencia del Boosting tradicional, XGBoost **castiga a los árboles que se vuelven demasiado complejos**, lo que ayuda a evitar el sobreajuste (*overfitting*). Este castigo se incorpora directamente en la función objetivo.

**Paralelización de estructura:** Aunque el boosting es secuencial por naturaleza (un árbol depende del anterior), XGBoost es extremadamente rápido porque **paraleliza la búsqueda de los puntos de corte** (*splits*) dentro de cada nivel del árbol.

**Manejo de datos dispersos:** Tiene una «dirección por defecto» incorporada. Si encuentra un valor nulo, ya sabe hacia qué lado de la rama enviarlo, basándose en la reducción de la pérdida. Aprende automáticamente cómo manejar información faltante.

> **Analogía de clase:** Si Random Forest es una democracia y Gradient Boosting es un aprendizaje secuencial meritocrático, XGBoost es ese mismo aprendizaje pero con un reglamento estricto que penaliza la complejidad excesiva.

---
# 2. Fundamentos Matemáticos

## 2.1 Función Objetivo con Regularización

La diferencia fundamental de XGBoost es que optimiza una función objetivo que incluye un **término de regularización** $\Omega$:

$$\text{Obj}(\phi) = \sum_{i} L(y_i, \hat{y}_i) + \sum_{k} \Omega(f_k)$$

donde $f_k$ son los árboles del ensemble. El primer término mide el **error de ajuste** y el segundo penaliza la **complejidad del modelo**.

## 2.2 Término de Regularización $\Omega$

La regularización se define como:

$$\Omega(f) = \gamma T + \frac{1}{2}\lambda\|\omega\|^2$$

donde:
- $T$ = número de hojas del árbol (penaliza árboles con muchas hojas)
- $\omega$ = pesos (valores) de las hojas
- $\gamma$ = umbral mínimo de ganancia para hacer una división
- $\lambda$ = fuerza de la regularización L2 (controla la magnitud de los pesos)

## 2.3 Aproximación de Taylor de Segundo Orden

Para ser más rápido y preciso, XGBoost aproxima la función de pérdida usando la **expansión de Taylor de segundo orden**. Esto le permite usar no solo el gradiente sino también el Hessiano:

$$L(y_i, \hat{y}_i^{(t)}) \approx L(y_i, \hat{y}_i^{(t-1)}) + g_i f_t(x_i) + \frac{1}{2} h_i f_t(x_i)^2$$

donde:
- $g_i = \dfrac{\partial L(y_i, \hat{y}_i)}{\partial \hat{y}_i}$ = **gradiente** (primera derivada de la pérdida)
- $h_i = \dfrac{\partial^2 L(y_i, \hat{y}_i)}{\partial \hat{y}_i^2}$ = **Hessiano** (segunda derivada de la pérdida)

Para MSE específicamente:
$$g_i = -(y_i - f_m) \qquad h_i = 1 \text{ (constante)}$$

## 2.4 Peso Óptimo de una Hoja (con Regularización L2)

El valor óptimo que debe tener cada hoja, considerando la regularización, es:

$$\omega^* = -\frac{\sum_i g_i}{\sum_i h_i + \lambda}$$

Para MSE simplificado (donde $h_i = 1$ para todas las muestras):

$$\omega^* = \frac{\sum_i r_i}{n + \lambda}$$

Cuando $\lambda = 0$, recuperamos el promedio de residuos puro del Gradient Boosting. Cuando $\lambda > 0$, el peso se **encoge** hacia cero, evitando sobreajuste.

## 2.5 Ganancia de Estructura (Score de División)

Para decidir si vale la pena hacer una división, XGBoost calcula la **Ganancia de Estructura**:

$$\text{Gain} = \frac{1}{2}\left[\frac{(\sum g_L)^2}{\sum h_L + \lambda} + \frac{(\sum g_R)^2}{\sum h_R + \lambda} - \frac{(\sum g_L + g_R)^2}{\sum h_L + h_R + \lambda}\right] - \gamma$$

donde $L$ y $R$ son las ramas izquierda y derecha de la división.

- Se elige la división que **maximice** el Gain.
- Si el Gain máximo es **menor a 0**, se realiza **poda** (*pruning*) y no se divide el nodo.

## 2.6 Métricas de Complejidad

| Métrica | Complejidad | Notas |
|---|---|---|
| **Entrenamiento** | $O(M \cdot d \cdot n\log n)$ | Gracias a *block structures* y cómputo fuera de memoria (*out-of-core*), es drásticamente más veloz que GBM estándar |
| **Predicción** | $O(M \cdot \text{profundidad})$ | Igual que Gradient Boosting: suma las predicciones de los $M$ árboles |
| **Espacio** | $O(M \cdot \text{nodos}) + O(n \cdot d)$ | Espacio para los árboles **más** los índices de características ordenadas (histogramas) |

---
# 3. Pseudocódigo

```
Algoritmo: XGBoost (Nivel de Nodo)
==========================================================
ENTRADA:
    D              → dataset (X, y)
    M              → número de árboles
    η              → tasa de aprendizaje
    λ              → término de regularización L2
    max_depth      → profundidad máxima de cada árbol

SALIDA:
    F_M(x)         → modelo final regularizado

----------------------------------------------------------
PASO 1 — Inicializar:
    F_0 ← media(y)
    f_acumulado ← vector lleno con F_0

PASO 2 — Para m = 1 hasta M:

    2a. Calcular gradientes y hessianos para cada muestra:
            residuos = y - f_acumulado
            gᵢ = -(yᵢ - f_acumulado[i])   [para MSE]
            hᵢ = 1                          [para MSE, constante]

    2b. Construir árbol regularizado sobre los residuos:
        PARA cada posible división (split) en cada feature:
            1. Calcular Ganancia de Estructura:
               Gain = ½ [ (ΣgL)²/(ΣhL+λ) + (ΣgR)²/(ΣhR+λ)
                          - (ΣgL+gR)²/(ΣhL+hR+λ) ] - γ
            2. Elegir la división que maximice el Gain
            3. Si Gain máximo < 0 → NO dividir (poda)

        En cada hoja, asignar peso regularizado:
            ω* = Σgᵢ / (Σhᵢ + λ)
            Para MSE: ω* = Σresidуos / (n_hoja + λ)

    2c. Actualizar modelo:
            f_acumulado += η * árbol_m.predict(X)

    2d. Guardar árbol_m en lista_arboles

PASO 3 — Predicción para X_nuevo:
    y_pred ← vector lleno con F_0
    PARA cada árbol en lista_arboles:
        y_pred += η * árbol.predict(X_nuevo)
    RETORNAR y_pred

==========================================================
DIFERENCIA CLAVE vs Gradient Boosting estándar:
----------------------------------------------------------
  GB estándar → valor hoja = media(residuos)
  XGBoost     → valor hoja = Σresidуos / (n_hoja + λ)
                             ← el denominador λ "encoge" el peso
                                hacia cero, evitando sobreajuste
```

---
# 4. Código

In [ ]:
# ============================================================
# IMPORTACIONES
# ============================================================
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
# ============================================================
# CLASE: Nodo
# Unidad básica del árbol de decisión.
# ============================================================
class Nodo:
    """
    Representa un nodo dentro de un árbol de decisión.

    Parámetros
    ----------
    feature   : int   → índice del feature sobre el que se divide
    threshold : float → valor umbral de la división
    left      : Nodo  → subárbol izquierdo  (X[:, feature] <= threshold)
    right     : Nodo  → subárbol derecho    (X[:, feature]  > threshold)
    value     : float → predicción si es hoja (peso regularizado ω*)
    """
    def __init__(self, feature=None, threshold=None,
                 left=None, right=None, value=None):
        self.feature   = feature
        self.threshold = threshold
        self.left      = left
        self.right     = right
        self.value     = value

    def es_hoja(self):
        """Retorna True si el nodo es una hoja (sin hijos)."""
        return self.value is not None

In [ ]:
# ============================================================
# CLASE: ArbolDecisionRegresor  (árbol base — igual que en GB)
# Criterio de impureza: Varianza Ponderada (MSE).
# XGBoost hereda de esta clase y sobreescribe el valor de hoja.
# ============================================================
class ArbolDecisionRegresor:
    """
    Árbol de decisión regresor base.
    Usado directamente por Gradient Boosting y como clase
    padre de ArbolDecisionRegresorRegularizado (XGBoost).

    Parámetros
    ----------
    max_depth : int → profundidad máxima del árbol (default: 3)
    """

    def __init__(self, max_depth=3):
        self.max_depth = max_depth
        self.root      = None

    def fit(self, X, y):
        """
        Construye el árbol recursivamente.

        Parámetros
        ----------
        X : np.ndarray (n_muestras, n_features)
        y : np.ndarray (n_muestras,) → residuos a predecir
        """
        self.root = self._crear_arbol(X, y, depth=0)

    def _crear_arbol(self, X, y, depth):
        """Crea nodos recursivamente."""
        n_muestras = len(y)

        # Condición de parada → hoja con promedio de residuos
        if depth >= self.max_depth or n_muestras < 2:
            return Nodo(value=np.mean(y))

        mejor_feat, mejor_umbral = self._mejor_criterio(X, y)

        if mejor_feat is None:
            return Nodo(value=np.mean(y))

        mascara_izq = X[:, mejor_feat] <= mejor_umbral
        mascara_der = ~mascara_izq

        hijo_izq = self._crear_arbol(X[mascara_izq], y[mascara_izq], depth + 1)
        hijo_der = self._crear_arbol(X[mascara_der], y[mascara_der], depth + 1)

        return Nodo(feature=mejor_feat, threshold=mejor_umbral,
                    left=hijo_izq, right=hijo_der)

    def _mejor_criterio(self, X, y):
        """Encuentra el feature y umbral que minimizan la varianza ponderada (MSE)."""
        mejor_mse    = np.var(y)
        mejor_feat   = None
        mejor_umbral = None

        for idx_feat in range(X.shape[1]):
            for umbral in np.unique(X[:, idx_feat]):
                y_izq = y[X[:, idx_feat] <= umbral]
                y_der = y[X[:, idx_feat] >  umbral]

                if len(y_izq) == 0 or len(y_der) == 0:
                    continue

                mse_ponderado = (
                    len(y_izq) * np.var(y_izq) +
                    len(y_der) * np.var(y_der)
                ) / len(y)

                if mse_ponderado < mejor_mse:
                    mejor_mse    = mse_ponderado
                    mejor_feat   = idx_feat
                    mejor_umbral = umbral

        return mejor_feat, mejor_umbral

    def predict(self, X):
        """Predice un valor continuo para cada muestra en X."""
        return np.array([self._recorrer_arbol(x, self.root) for x in X])

    def _recorrer_arbol(self, x, nodo):
        """Recorre el árbol para una sola muestra x."""
        if nodo.es_hoja():
            return nodo.value
        if x[nodo.feature] <= nodo.threshold:
            return self._recorrer_arbol(x, nodo.left)
        return self._recorrer_arbol(x, nodo.right)

In [ ]:
# ============================================================
# CLASE: ArbolDecisionRegresorRegularizado
# Extiende ArbolDecisionRegresor para aplicar regularización L2
# en el valor de cada hoja — ésta es la esencia de XGBoost.
# ============================================================
class ArbolDecisionRegresorRegularizado(ArbolDecisionRegresor):
    """
    Árbol regresor con peso de hoja regularizado por L2.

    La única diferencia respecto al árbol base es el valor
    que se asigna a las hojas:

        GB estándar : valor_hoja = media(residuos)
        XGBoost     : valor_hoja = Σresidуos / (n_hoja + λ)

    Cuando λ > 0, el peso se «encoge» hacia cero, lo que
    reduce la varianza y controla el sobreajuste.

    Parámetros
    ----------
    max_depth  : int   → profundidad máxima (default: 3)
    reg_lambda : float → fuerza de regularización L2 λ (default: 1.0)
    """

    def __init__(self, max_depth=3, reg_lambda=1.0):
        super().__init__(max_depth)    # hereda max_depth y root de la clase padre
        self.reg_lambda = reg_lambda

    def _crear_arbol(self, X, y, depth):
        """
        Sobreescribe la condición de parada para aplicar
        la fórmula de hoja regularizada de XGBoost.

        Fórmula del peso óptimo:
            ω* = Σgᵢ / (Σhᵢ + λ)
        Para MSE (hᵢ = 1):
            ω* = Σresidуos / (n_hoja + λ)
        """
        if depth >= self.max_depth or len(y) < 2:
            # Peso regularizado: ω* = sum(g) / (sum(h) + λ)
            peso_regularizado = np.sum(y) / (len(y) + self.reg_lambda)
            return Nodo(value=peso_regularizado)

        # Para nodos internos, reutiliza el criterio de división del padre
        return super()._crear_arbol(X, y, depth)

In [ ]:
# ============================================================
# CLASE: XGBoostMiniDesdeCero
# Gradient Boosting con regularización L2 en las hojas.
# Implementa el alma de XGBoost: penalizar complejidad del árbol.
# ============================================================
class XGBoostMiniDesdeCero:
    """
    Implementación simplificada de XGBoost desde cero.

    Extiende Gradient Boosting añadiendo regularización L2 (λ)
    sobre el peso de las hojas de cada árbol estimador.

    Parámetros
    ----------
    n_estimators  : int   → número de árboles (default: 10)
    learning_rate : float → tasa de aprendizaje η (default: 0.1)
    max_depth     : int   → profundidad máxima de cada árbol (default: 3)
    reg_lambda    : float → fuerza de regularización L2 λ (default: 1.0)
                            λ=0 → igual que GB estándar
                            λ>0 → pesos de hojas más pequeños (menos sobreajuste)

    Atributos (tras fit)
    --------------------
    init_prediction : float       → predicción base = media(y)
    trees           : list[Árbol] → árboles regularizados entrenados
    """

    def __init__(self, n_estimators=10, learning_rate=0.1,
                 max_depth=3, reg_lambda=1.0):
        self.n_estimators  = n_estimators
        self.learning_rate = learning_rate
        self.max_depth     = max_depth
        self.reg_lambda    = reg_lambda   # Término de regularización L2
        self.trees         = []
        self.init_prediction = 0

    # ----------------------------------------------------------
    # ENTRENAMIENTO
    # ----------------------------------------------------------
    def fit(self, X, y):
        """
        Entrena el modelo XGBoost.

        En cada iteración:
          · Calcula gradientes (g) y hessianos (h) para MSE
          · Entrena un árbol regularizado sobre los residuos
          · Actualiza el modelo acumulado

        Parámetros
        ----------
        X : np.ndarray (n_muestras, n_features)
        y : np.ndarray (n_muestras,) → valores objetivo reales
        """
        # PASO 1: Predicción inicial = media(y)
        self.init_prediction = np.mean(y)
        f_acumulado = np.full(y.shape, self.init_prediction)

        for _ in range(self.n_estimators):
            # Calcular gradientes (g) y hessianos (h)
            # Para MSE: g = -(y - f_m),  h = 1 (constante)
            residuos = y - f_acumulado

            # Entrenar árbol que considera la regularización λ
            arbol = ArbolDecisionRegresorRegularizado(
                max_depth=self.max_depth,
                reg_lambda=self.reg_lambda
            )
            arbol.fit(X, residuos)

            # Actualizar modelo: F_m = F_{m-1} + η · h_m(x)
            f_acumulado += self.learning_rate * arbol.predict(X)
            self.trees.append(arbol)

    # ----------------------------------------------------------
    # PREDICCIÓN
    # ----------------------------------------------------------
    def predict(self, X):
        """
        Genera predicciones para nuevas muestras.

        Parámetros
        ----------
        X : np.ndarray (n_muestras, n_features)

        Retorna
        -------
        np.ndarray con las predicciones finales
        """
        y_pred = np.full(X.shape[0], self.init_prediction)
        for arbol in self.trees:
            y_pred += self.learning_rate * arbol.predict(X)
        return y_pred

---
# 5. Verificación del Código (Ejemplo de Clase)

Se reproduce el ejemplo del notebook de clase: ajuste de XGBoost a una función no lineal (seno con ruido).

In [ ]:
# ============================================================
# VERIFICACIÓN — Ejemplo del notebook de clase
# Datos: curva seno con ruido gaussiano (función no lineal)
# ============================================================

np.random.seed(42)
X_seno = np.sort(5 * np.random.rand(80, 1), axis=0)
y_seno = np.sin(X_seno).ravel() + np.random.normal(0, 0.1, X_seno.shape[0])

# Instanciar y entrenar — mismos parámetros que en clase
modelo_clase = XGBoostMiniDesdeCero(
    n_estimators=30,
    learning_rate=0.2,
    max_depth=3,
    reg_lambda=1.0
)
modelo_clase.fit(X_seno, y_seno)

# Predicción sobre curva continua para visualizar
X_curva = np.arange(0.0, 5.0, 0.01)[:, np.newaxis]
y_curva = modelo_clase.predict(X_curva)

# Gráfica — igual a la del notebook de clase
plt.figure(figsize=(10, 6))
plt.scatter(X_seno, y_seno, color='tab:gray', alpha=0.5, label='Datos con Ruido (Reales)')
plt.plot(X_curva, y_curva, color='tab:red', linewidth=2, label='Predicción XGBoost (Fitted)')
plt.title('Prueba de XGBoost: Ajuste a Función No Lineal')
plt.xlabel('Característica X')
plt.ylabel('Objetivo Y')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.7)
plt.show()

---
# 6. Aplicación con Nuevo Input

> **Instrucción para el examen:** Sustituir los datos de la celda siguiente con el input que proporcione la Dra. Méndez Rincón. El resto del código (modelo, entrenamiento, predicción y métricas) se ejecuta sin modificaciones.

In [ ]:
# ============================================================
# >>>  NUEVO INPUT — REEMPLAZAR CON LOS DATOS DEL EXAMEN  <<<
# ============================================================

# --- DATOS DE ENTRENAMIENTO ---
# X_train debe ser un arreglo 2D: shape (n_muestras, n_features)
# y_train debe ser un arreglo 1D: shape (n_muestras,)

X_train = np.array([   # <-- reemplazar
    [1],
    [2],
    [3],
    [4],
    [5]
])

y_train = np.array([2, 4, 6, 8, 10], dtype=float)  # <-- reemplazar

# --- DATOS DE PRUEBA ---
X_test = np.array([[6]])  # <-- reemplazar

# --- HIPERPARÁMETROS ---
# Ajustar si la maestra especifica valores distintos
N_ESTIMATORS  = 30
LEARNING_RATE = 0.2
MAX_DEPTH     = 3
REG_LAMBDA    = 1.0   # λ: 0 = sin regularización, >0 = más control del sobreajuste

In [ ]:
# ============================================================
# ENTRENAMIENTO Y PREDICCIÓN
# (No modificar — funciona con cualquier input de arriba)
# ============================================================

# Instanciar y entrenar modelo
modelo = XGBoostMiniDesdeCero(
    n_estimators=N_ESTIMATORS,
    learning_rate=LEARNING_RATE,
    max_depth=MAX_DEPTH,
    reg_lambda=REG_LAMBDA
)
modelo.fit(X_train, y_train)

# Predicciones sobre entrenamiento
y_pred_train = modelo.predict(X_train)

# Predicción sobre el nuevo input
y_pred_test = modelo.predict(X_test)

print("=" * 55)
print(" PREDICCIONES EN ENTRENAMIENTO")
print("=" * 55)
for i in range(len(y_train)):
    error = abs(y_train[i] - y_pred_train[i])
    print(f"  Real: {y_train[i]:8.2f}  |  Predicho: {y_pred_train[i]:8.2f}  |  Error: {error:8.4f}")

print("\n" + "=" * 55)
print(" PREDICCIÓN PARA NUEVO INPUT")
print("=" * 55)
for i, pred in enumerate(y_pred_test):
    print(f"  X_test[{i}] = {X_test[i]}  →  Predicción: {pred:.4f}")

In [ ]:
# ============================================================
# MÉTRICAS DE DESEMPEÑO
# ============================================================

# MSE — Error Cuadrático Medio
mse = np.mean((y_train - y_pred_train) ** 2)

# RMSE — Raíz del Error Cuadrático Medio (mismas unidades que y)
rmse = np.sqrt(mse)

# MAE — Error Absoluto Medio
mae = np.mean(np.abs(y_train - y_pred_train))

# R² — Coeficiente de determinación
ss_res = np.sum((y_train - y_pred_train) ** 2)
ss_tot = np.sum((y_train - np.mean(y_train)) ** 2)
r2 = 1 - (ss_res / ss_tot)

print("=" * 45)
print(" MÉTRICAS DE DESEMPEÑO (train)")
print("=" * 45)
print(f"  MSE  (Error Cuadrático Medio):    {mse:.4f}")
print(f"  RMSE (Raíz del MSE):              {rmse:.4f}")
print(f"  MAE  (Error Absoluto Medio):      {mae:.4f}")
print(f"  R²   (Coeficiente determinación): {r2:.4f}")
print("\nInterpretación:")
print(f"  → R² = {r2:.4f}: el modelo explica el {r2*100:.1f}% de la varianza en y.")
print(f"  → RMSE = {rmse:.4f}: error promedio en las mismas unidades que y.")

In [ ]:
# ============================================================
# VISUALIZACIÓN — Efecto de λ (reg_lambda) sobre las predicciones
# Muestra la diferencia clave entre XGBoost y GB estándar
# ============================================================

lambdas      = [0.0, 1.0, 5.0, 20.0]
colores      = ['steelblue', 'darkorange', 'tomato', 'purple']
y_pred_train_list = []

plt.figure(figsize=(10, 5))

for lam, color in zip(lambdas, colores):
    m = XGBoostMiniDesdeCero(
        n_estimators=N_ESTIMATORS,
        learning_rate=LEARNING_RATE,
        max_depth=MAX_DEPTH,
        reg_lambda=lam
    )
    m.fit(X_train, y_train)

    # Predicción sobre rango continuo para curva suave
    X_rango = np.linspace(X_train.min(), X_train.max(), 200).reshape(-1, 1)
    y_rango  = m.predict(X_rango)
    etiqueta = f'λ = {lam}' + (' (sin regularización)' if lam == 0 else '')
    plt.plot(X_rango, y_rango, color=color, linewidth=2, label=etiqueta)

plt.scatter(X_train, y_train, color='black', zorder=5, label='Datos reales', s=50)
plt.xlabel('X')
plt.ylabel('y predicho')
plt.title('Efecto de la Regularización λ en XGBoost\nλ alto → modelo más suave, menos sobreajuste')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Interpretación:")
print("  · λ = 0.0 → sin regularización, equivale a Gradient Boosting estándar.")
print("  · λ grande → pesos de hojas más pequeños, curva más suave, menos sobreajuste.")
print("  · λ = 1.0 es el valor por defecto recomendado en la práctica.")

<hr/>
<footer style='text-align:center; font-size:12px; color:gray;'>
© 2026 UNAM Facultad de Ciencias – Todos los derechos reservados
</footer>